# Numérique-01 — Équation de Burgers : volumes finis, solveur de Riemann, et ce qu'une preuve prouve

[↑ Numérique](README.md) | [Version Python](Numerique-01-Burgers-Volumes-Finis-Python.ipynb)

## Objectifs pédagogiques

1. Dériver la condition de Rankine–Hugoniot et la solution exacte du problème de Riemann pour Burgers.
2. Implémenter les flux de Godunov et de Rusanov et un schéma volumes finis sous condition CFL.
3. Mesurer l'erreur L1 et le taux de convergence sur un choc et une raréfaction.
4. Mesurer la formation d'un choc par les caractéristiques.
5. Distinguer ce que les mesures établissent de ce qui demanderait une preuve formelle.

**Prérequis** : C# de base, dérivées partielles et intégration. Aucun paquet NuGet ; .NET Interactive et bibliothèque standard uniquement.

**Durée estimée** : 45–60 minutes. Expérience CPU déterministe.

In [1]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Kernel C# actif : .NET {Environment.Version}; bibliothèque standard uniquement.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Kernel C# actif : .NET 10.0.13; bibliothèque standard uniquement.


## Partie 1 — Flux et solveur exact de Riemann

Pour $f(u)=u^2/2$, la vitesse du choc est $s=(u_L+u_R)/2$. Si $u_L<u_R$, la solution est un éventail de raréfaction ; sinon c'est un choc. Les fonctions suivantes évaluent le flux et l'état au centre de l'éventail.

In [2]:
double Flux(double u) => 0.5 * u * u;
double RiemannExact(double ul, double ur, double x, double t)
{
    if (t <= 0) return x < 0 ? ul : ur;
    if (ul > ur) return x < 0.5 * (ul + ur) * t ? ul : ur;
    double xi = x / t;
    return xi <= ul ? ul : xi >= ur ? ur : xi;
}
double EtatEtoile(double ul, double ur)
{
    if (ul > ur) return 0.5 * (ul + ur) > 0 ? ul : ur;
    return ul >= 0 ? ul : ur <= 0 ? ur : 0;
}
double Godunov(double ul, double ur) => Flux(EtatEtoile(ul, ur));
double Rusanov(double ul, double ur)
{
    double a = Math.Max(Math.Abs(ul), Math.Abs(ur));
    return 0.5 * (Flux(ul) + Flux(ur)) - 0.5 * a * (ur - ul);
}
if (Math.Abs(Flux(1.2) - 0.72) > 1e-14 || Math.Abs(RiemannExact(0.4, 1.2, 0.25, 0.5) - 0.5) > 1e-14 || Math.Abs(EtatEtoile(1.2, 0.4) - 1.2) > 1e-14) throw new Exception("Contrôle Riemann échoué.");
Console.WriteLine($"Flux(1.2)={Flux(1.2):F2}; vitesse du choc (1.2,0.4)={0.5*(1.2+0.4):F2}; état de raréfaction au centre={RiemannExact(0.4,1.2,0.25,0.5):F2}.");

Flux(1.2)=0.72; vitesse du choc (1.2,0.4)=0.80; état de raréfaction au centre=0.50.


## Partie 2 — Volumes finis : Godunov et Rusanov

La moyenne cellulaire évolue selon $U_i^{n+1}=U_i^n-(\Delta t/\Delta x)(F_{i+1/2}-F_{i-1/2})$. Le pas respecte la condition CFL et est borné pour atteindre exactement le temps final. Les bords sont gelés, suffisamment éloignés des fronts pour ces expériences.

In [3]:
double[] Step(double[] u, double dx, double cfl, Func<double,double,double> flux, double dtMax)
{
    double vmax = u.Max(Math.Abs);
    double dt = Math.Min(cfl * dx / Math.Max(vmax, 1e-14), dtMax);
    var f = new double[u.Length - 1];
    for (int i=0;i<f.Length;i++) f[i]=flux(u[i],u[i+1]);
    var next=(double[])u.Clone();
    for(int i=1;i<u.Length-1;i++) next[i]=u[i]-(dt/dx)*(f[i]-f[i-1]);
    return next;
}
(double[] U, int Steps) Simulate(int n,double ul,double ur,double tFinal,Func<double,double,double> flux)
{
    const double xmin=-1, xmax=2, cfl=0.9;
    double dx=(xmax-xmin)/n; var u=new double[n];
    for(int i=0;i<n;i++) u[i]=xmin+(i+0.5)*dx<0?ul:ur;
    double t=0; int steps=0;
    while(t<tFinal-1e-14){double dt=Math.Min(cfl*dx/Math.Max(u.Max(Math.Abs),1e-14),tFinal-t);u=Step(u,dx,cfl,flux,dt);t+=dt;steps++;}
    return (u,steps);
}
double L1Error(int n,double ul,double ur,double tFinal,Func<double,double,double> flux)
{
    const double xmin=-1,xmax=2; double dx=(xmax-xmin)/n; var (u,_)=Simulate(n,ul,ur,tFinal,flux); double sum=0;
    for(int j=0;j<6001;j++){double x=xmin+(xmax-xmin)*j/6000.0;double numPos=(x-xmin)/dx-0.5;int i=(int)Math.Floor(numPos);double val=i<0?u[0]:i>=n-1?u[n-1]:u[i]+(numPos-i)*(u[i+1]-u[i]);sum+=Math.Abs(val-RiemannExact(ul,ur,x,tFinal));}
    return sum*(xmax-xmin)/6000.0;
}
var (ug,ng)=Simulate(400,1.2,0.4,0.5,Godunov);var (ur, nr)=Simulate(400,1.2,0.4,0.5,Rusanov);
double eg=L1Error(400,1.2,0.4,0.5,Godunov), er=L1Error(400,1.2,0.4,0.5,Rusanov);
if(ng!=89||nr!=89||!(eg>0&&er>eg))throw new Exception($"Schéma inattendu : pas {ng}/{nr}, erreurs {eg}/{er}.");
Console.WriteLine($"Choc (N=400, T=0.5) : {ng} pas communs; erreur L1 Godunov={eg:F6}, Rusanov={er:F6}.");

Choc (N=400, T=0.5) : 89 pas communs; erreur L1 Godunov=0.004478, Rusanov=0.005310.


### 2.2 Étude de convergence

On mesure l'erreur L1 par régression log-log sur les mêmes tailles de grille. Cette expérience corrobore les cas testés ; elle ne constitue pas une preuve de convergence.

In [4]:
double Pente(double[] x,double[] y){double mx=x.Average(),my=y.Average();return x.Zip(y,(a,b)=>(a-mx)*(b-my)).Sum()/x.Sum(a=>(a-mx)*(a-mx));}
double[] grids={100,200,400,800,1600};
double MesurePente(double ul,double ur,Func<double,double,double> flux){double[] dx=grids.Select(n=>3.0/n).ToArray();double[] err=grids.Select(n=>L1Error((int)n,ul,ur,0.5,flux)).ToArray();return Pente(dx.Select(v=>Math.Log(v)).ToArray(),err.Select(v=>Math.Log(v)).ToArray());}
double chocG=MesurePente(1.2,0.4,Godunov), chocR=MesurePente(1.2,0.4,Rusanov), fanG=MesurePente(0.4,1.2,Godunov), fanR=MesurePente(0.4,1.2,Rusanov);
if(chocG<0.8||chocG>1.2||chocR<0.8||chocR>1.2||fanG<0.5||fanG>1.2||fanR<0.5||fanR>1.2)throw new Exception("Pente de convergence hors des plages de contrôle.");
Console.WriteLine($"Ordres mesurés — choc : Godunov {chocG:F2}, Rusanov {chocR:F2}; raréfaction : Godunov {fanG:F2}, Rusanov {fanR:F2}.");

Ordres mesurés — choc : Godunov 0.99, Rusanov 0.99; raréfaction : Godunov 0.75, Rusanov 0.79.


## Partie 3 — Formation du choc par les caractéristiques

Pour $u_0(x)=1+\frac12\sin(2\pi x)$, le temps prédit est $t^*=1/\pi$. On estime aussi le croisement de particules voisines par dichotomie ; aucun schéma volumes finis n'intervient dans cette mesure.

In [5]:
double U0(double x)=>1+0.5*Math.Sin(2*Math.PI*x);
double Gap(double t,int n){double min=double.PositiveInfinity;double prev=0;for(int i=0;i<=n;i++){double x=(double)i/n, pos=x+U0(x)*t;if(i>0)min=Math.Min(min,pos-prev);prev=pos;}return min;}
int particles=2000;double lo=0,hi=1;for(int k=0;k<60;k++){double mid=0.5*(lo+hi);if(Gap(mid,particles)>0)lo=mid;else hi=mid;}
double measured=0.5*(lo+hi), predicted=1/Math.PI, relative=Math.Abs(measured-predicted)/predicted;
if(relative>1e-4)throw new Exception($"Temps de choc incohérent : {relative:G}.");
Console.WriteLine($"Temps de choc prédit 1/pi={predicted:F6}; mesure particules={measured:F6}; écart relatif={relative:E2}.");

Temps de choc prédit 1/pi=0.318310; mesure particules=0.318310; écart relatif=1.64E-006.


## Partie 4 — Ce que les mesures et les preuves établissent

Les expériences numériques mesurent les sorties de schémas sur un ensemble fini de cas. Elles ne démontrent pas leur convergence pour toute donnée. Le dépôt ne contient pas de preuve formelle de correction du schéma discret de Godunov ; la preuve de ce lien reste un grain ouvert. Les mesures reproduites ici sont déterministes et les sorties du carnet sont enregistrées après exécution.

## Vérifiez votre compréhension

Les quatre exercices du carnet Python — Lax–Wendroff, solveur de Riemann cubique, largeur de choc et entropie discrète — sont à traiter dans la version Python ; ce compagnon porte les expériences principales et conserve ici des exercices C# compilables.

In [6]:
// Exercice 1 — Étudier la largeur du choc. TODO étudiant.
double? largeurChoc = null;
Console.WriteLine(largeurChoc is null ? "Exercice 1 à compléter : compter les traversées du niveau milieu." : $"Largeur mesurée : {largeurChoc}");

Exercice 1 à compléter : compter les traversées du niveau milieu.


In [7]:
// Exercice 2 — Implémenter le solveur exact pour un flux cubique. TODO étudiant.
#nullable enable
Func<double,double,double,double,double>? riemannCubique = null;
Console.WriteLine(riemannCubique is null ? "Exercice 2 à compléter : solveur de Riemann pour f(u)=u³/3." : "Solveur cubique défini.");

Exercice 2 à compléter : solveur de Riemann pour f(u)=u³/3.


In [8]:
// Exercice 3 — Tracer l'entropie discrète jusqu'à T=0.5. TODO étudiant.
#nullable enable
List<double>? entropieDiscrete = null;
Console.WriteLine(entropieDiscrete is null ? "Exercice 3 à compléter : vérifier la décroissance de l'entropie." : $"{entropieDiscrete.Count} mesures.");

Exercice 3 à compléter : vérifier la décroissance de l'entropie.


## À retenir

| Notion | Résultat | Partie |
|---|---|---|
| Rankine–Hugoniot | $s=(u_L+u_R)/2$ ; pour (1.2, 0.4), $s=0.8$ | 1 |
| Godunov et Rusanov | erreurs L1 mesurées et comparées sur le même choc | 2 |
| Convergence | pentes log-log mesurées sur choc et raréfaction | 2 |
| Formation d'un choc | $t^*=1/\pi$, confronté aux caractéristiques | 3 |
| Frontière de preuve | mesures sur cas test, pas de preuve formelle du schéma | 4 |

## Pour aller plus loin

Consultez [la version Python](Numerique-01-Burgers-Volumes-Finis-Python.ipynb) pour les développements pédagogiques complets et les quatre exercices approfondis ; puis [Numérique-02 — Maxwell en relativité](Numerique-02-Maxwell-Relativiste-Volumes-Finis-C.ipynb).